In [ ]:
folder_path = r"C:\Users\HP\global"

In [ ]:
import os
import glob


files = glob.glob(os.path.join(folder_path, "*.csv"))

len(files)

**Note:** Folder mein **6 CSV files** mili - har saal (2020–2025) ki ek file, yaani data already saal-wise split hai aur ek hi structure follow karta hai.

In [ ]:
import pandas as pd

dataframes = []

for file in files:
    df_temp = pd.read_csv(file)
    dataframes.append(df_temp)

combined_df = pd.concat(dataframes, ignore_index=True)

**Note:** 6 files ko `pd.concat` se ek hi DataFrame `combined_df` mein jod diya (aage ka saara kaam isi par). `ignore_index=True` se index 0 se continuous reset hua.

In [ ]:
combined_df.shape

**Note:** Combined data = **27,77,070 rows × 17 columns** (~2.78 million price records). Saal-wise rows: 2020 to 4.38 lakh, 2021 to 4.92 lakh, 2022 to **5.01 lakh (peak)**, 2023 to 4.89 lakh, 2024 to 4.37 lakh, 2025 to 4.20 lakh. Yaani 2022 ke baad coverage thoda ghata hai (countries 87 to 72), jo trend analysis mein dhyan rakhna hai.

In [ ]:
for file in files:
    print(os.path.basename(file))

**Note:** Files ke naam consistent hain (`wfp_food_prices_global_<year>.csv`) aur 2020 se 2025 tak sorted order mein padhi gayi - koi saal missing nahi hai.

In [ ]:
combined_df.columns

**Note:** 17 columns 4 groups mein baanti ja sakti hain: **Location** (countryiso3, admin1, admin2, market, market_id, latitude, longitude) · **Product** (category, commodity, commodity_id, unit) · **Price info** (priceflag, pricetype, currency, price, usdprice) · **Time** (date). `price` local currency mein hai jabki `usdprice` global comparison ke liye - analysis ka main column `usdprice` + `currency` hoga.

In [ ]:
combined_df.dtypes

**Note:** `date` abhi **string** hai (isse time analysis nahi ho sakti) to datetime mein convert karna zaroori. `price` aur `usdprice` sahi se **float64** hain, ids **int64**, baaki text columns.

In [ ]:
combined_df['date'] = pd.to_datetime(
    combined_df['date'],
    format='mixed',
    errors='coerce'
)

**Note:** `format='mixed'` se alag-alag date formats wali files bhi ek saath datetime mein convert ho gayin (`errors='coerce'` ke saath, taaki koi galat date NaT ban jaye, crash na ho).

In [ ]:
combined_df['date'].dt.strftime('%Y-%m-%d').head(10)

**Note:** Saari dates **15 tareekh** ki hain (jaise 2020-01-15) - matlab data **monthly** hai, har mahine ka ek mid-month price point. Isliye aage month-level (`ym`) par group karna sahi rahega.

In [ ]:
combined_df['date'].isnull().sum()

**Note:** **0 null dates** - kisi bhi row ki date convert karne mein dikkat nahi aayi. Date range **15 Jan 2020 se 15 Dec 2025** = **72 months** ka poora data.

In [ ]:
(combined_df.isnull().sum() / len(combined_df) * 100).round(3)

**Note:** Data bahut complete hai: sirf `admin1`, `admin2`, `latitude`, `longitude` mein **0.166%** nulls (~4,601 rows) aur `usdprice` mein **0.158%** (4,375 rows - zyada tar **Somalia 3,975** aur **Moldova 400**). Baaki 12 columns mein **0% null**.

In [ ]:
combined_df['admin1'] = combined_df['admin1'].fillna('Unknown')

combined_df['admin2'] = combined_df['admin2'].fillna('Unknown')

**Note:** `admin1`/`admin2` ke 0.166% nulls ko `'Unknown'` se bhara taaki in rows ko drop na karna pade (regional analysis mein baad mein `Unknown` ko alag rakha gaya).

In [ ]:
combined_df[['admin1', 'admin2']].isnull().sum()

**Note:** Ab `admin1` aur `admin2` mein **0 nulls** bache - fill sahi se ho gaya.

In [ ]:
combined_df.duplicated().sum()

**Note:** **0 duplicate rows** - data mein repeated records nahi hain, isliye dedupe ki zarurat nahi.

In [ ]:
combined_df.describe()

**Note:**
- `price` (local currency): **mean 12,514 vs median 500**, max **15.77 crore** to alag-alag currencies (SYP, LBP, ZWL...) mix hain, isliye `price` ko countries ke beech compare nahi kar sakte.
- `usdprice`: **median 1.20 USD**, 75% rows **≤ 2.56 USD**, par **mean 7.96** aur max **2,76,518 USD** to bahut zyada right-skewed, kuch rows pakka galat (unit/decimal errors).
- Date: 2020-01-15 to 2025-12-15 (median 2022-11-15); 4,405 unique markets.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

**Note:** `seaborn` aur `matplotlib` import hue - aage boxplots ke liye.

In [ ]:
Q1 = combined_df['price'].quantile(0.25)
Q3 = combined_df['price'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = combined_df[
    (combined_df['price'] < lower) |
    (combined_df['price'] > upper)
]

print("Total Outliers:", len(outliers))

**Note:** IQR method se **5,26,344 outliers (18.96%)** mile - par yeh **misleading** hai: `price` alag-alag currencies mein hai (4,000 ZWL aur 4,000 USD ek jaise nahi), isliye IQR yahan sahi tool nahi. Outlier check `usdprice` aur country-commodity ke apne median ke ratio par karna better hai.

In [ ]:
numeric_cols = ['price', 'usdprice']

for col in numeric_cols:
    plt.figure(figsize=(10, 4))

    sns.boxplot(x=combined_df[col])

    plt.title(f'Boxplot of {col}')
    plt.xlabel(col)

    plt.xticks(rotation=45)
    plt.show()

**Note:** `price` aur `usdprice` dono ke box zero ke paas pichak gaye hain aur ek lambi tail hai (price **1.58 × 10⁸** tak, usdprice **2.77 × 10⁵** tak) - normal scale par box padhna mushkil hai, isliye log scale chahiye.

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt

plt.figure(figsize=(10, 3))
sns.boxplot(x=combined_df['usdprice'], flierprops={'markersize': 1})
plt.xscale('log')
plt.title('Boxplot of usdprice (log scale)')
plt.show()

**Note:** Log scale par picture saaf hai: zyada tar `usdprice` **0.65 se 2.56 USD** ke beech hai (box), lekin dono taraf bahut saare points dikh rahe hain - upar ki taraf kuch **1,000 USD+** jaise impossible values hain jo `kg` food ke liye realistic nahi.

In [ ]:
s = combined_df['usdprice']
q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr

print("Q1:", round(q1, 2), "| Q3:", round(q3, 2), "| Upper fence:", round(upper, 2))
n = (s > upper).sum()
print("Above fence:", n, f"({n / len(s) * 100:.1f}%)")
print("Above 1000 USD:", (s > 1000).sum())

**Note:** Q1 = **0.65**, Q3 = **2.56**, upper fence = **5.43 USD**. Fence se upar **3,28,301 rows (11.8%)** - itne saare outlier nahi ho sakte, kyunki meat/dairy/oil jaise items sach mein 5+ USD/kg hote hain. Sirf **1,550 rows 1,000 USD se upar** hain, jo pakka galat hain. Yaani rule fixed fence nahi, **group-wise median ratio** hona chahiye.

In [ ]:
# 1. Sirf KG rakho (75% data, kaafi hai)
df = combined_df[combined_df['unit'] == 'KG'].copy()

# 2. usdprice missing wali rows hatao
df = df.dropna(subset=['usdprice'])

# 3. Pakka galat rows flag karo (delete nahi)
df['grp_median'] = df.groupby(['countryiso3','commodity'])['usdprice'].transform('median')
df['is_outlier'] = (df['usdprice'] / df['grp_median'] > 50) | (df['usdprice'] / df['grp_median'] < 0.02)

print(len(df), df['is_outlier'].sum())
df = df[~df['is_outlier']]

**Note:** `KG` unit **75.2%** data hai (baaki L, pcs, bundle jaise units - unka seedha comparison nahi ho sakta). `usdprice` null hatane ke baad **20,86,152 rows**. Rule (country-commodity median ka **>50x ya <0.02x**) ne **1,407 rows (0.07%)** flag ki aur hata di to **20,84,745 rows** bachi.

In [ ]:
outlier_df = df[df['is_outlier']]

outlier_df[
    ['countryiso3', 'commodity', 'date', 'usdprice', 'grp_median']
].head(20)

**Note:** Table **khali** aayi - kyunki cell 20 mein flagged rows pehle hi `df` se hata di gayi thi, to `df[df['is_outlier']]` mein kuch bacha hi nahi. Flagged rows dekhne ke liye pehle flag karo, **dekho**, phir hatao (isi wajah se cell 23 mein dobara flag kiya).

In [ ]:
print("Outlier rows:", len(outlier_df))
print("Flagged values:", df['is_outlier'].value_counts())

**Note:** `Outlier rows: 0` aur sab rows `False` - yeh confirm karta hai ki removal ho chuka hai, par yeh **inspection nahi** hai (same reason: flagged rows pehle hi hat gayi thi).

In [ ]:
df = combined_df[combined_df['unit'] == 'KG'].copy()

df = df.dropna(subset=['usdprice'])

df['grp_median'] = df.groupby(
    ['countryiso3', 'commodity']
)['usdprice'].transform('median')

df['is_outlier'] = (
    (df['usdprice'] / df['grp_median'] > 50) |
    (df['usdprice'] / df['grp_median'] < 0.02)
)

print("Total Records:", len(df))
print("Flagged Outliers:", df['is_outlier'].sum())

**Note:** Bina delete kiye dobara flag kiya: **20,86,152 records** mein se **1,407 (0.067%)** outlier flag hue - bahut kam hissa, yaani data overall saaf hai.

In [ ]:
outlier_df = df[df['is_outlier']]

display(outlier_df[
    ['countryiso3', 'commodity', 'date', 'usdprice', 'grp_median']
].head(20))

**Note:** Do tarah ke cases dikhte hain:
- **Asli galat values:** Uganda Maize **0.00057 USD** vs median 0.36; Tanzania Green beans **1,296 USD** vs median 1.42; Ethiopia Sugar **198 USD** vs 1.16 - decimal/unit error.
- **Zimbabwe (ZWE) ki rows:** USD prices **0.66–9.25** theek hain, par group median sirf **0.0009–0.0175** hai - yaani median hi kharab ho gaya (ZWL conversion se).

In [ ]:
outlier_df = df[df['is_outlier']].copy()

outlier_df['outlier_type'] = 'Normal'

outlier_df.loc[
    outlier_df['usdprice'] / outlier_df['grp_median'] > 50,
    'outlier_type'
] = 'High'

outlier_df.loc[
    outlier_df['usdprice'] / outlier_df['grp_median'] < 0.02,
    'outlier_type'
] = 'Low'

print(outlier_df['outlier_type'].value_counts())

**Note:** **1,351 High (96%)** vs **56 Low (4%)** - zyada tar flags upar ki taraf ke (bahut mehnge) galat values hain.

In [ ]:
high_outliers = outlier_df[
    outlier_df['outlier_type'] == 'High'
]

display(
    high_outliers.sort_values(
        'usdprice', ascending=False
    )[
        ['countryiso3', 'commodity', 'date',
         'usdprice', 'grp_median']
    ].head(20)
)

**Note:** Gambia **Mangoes 12,718.60 USD** vs median 0.52 (~24,000x) - possible typo ya unit issue; Tanzania Beans **1,296 USD**; Sudan **Wheat flour 327.69 USD** vs 2.40 aur **Sugar 322.96 USD** (Sept 2025 - currency collapse ya unit issue ho sakta hai, verify karna padega); Ethiopia Sugar **198 USD**. Kapenta fish (ZWE) bhi 300–800 USD tak - woh actually ZWL conversion problem hai.

In [ ]:
outlier_df['countryiso3'].value_counts().head(10)

**Note:** **1,407 mein se 1,323 (94%) flags sirf Zimbabwe** ki hain; baaki: Myanmar 20, Sudan 15, Cameroon 12, Benin 8, Fiji 6, Ethiopia 5. Matlab asli galat entries sirf **~84 rows** hain - Zimbabwe ka masla **currency mix** ka hai, data errors ka nahi.

In [ ]:
zwe_outliers = outlier_df[
    outlier_df['countryiso3'] == 'ZWE'
]

display(
    zwe_outliers[
        ['date', 'commodity', 'market', 'currency',
         'price', 'usdprice', 'grp_median']
    ].sort_values('date').head(20)
)

**Note:** Zimbabwe mein do currencies ek saath hain: **USD** rows (jaise Rice 1.34 USD) bilkul theek, jabki **ZWL** rows mein conversion bigda hua hai - jaise Rice **1.00 ZWL = 0.000015 USD**, lekin Fish (kapenta) **1,50,000 ZWL = 2.26 USD**. Isliye group median ZWL + USD mix hokar galat ban gaya.

In [ ]:
zwe_outliers.groupby('currency').agg(
    Records=('usdprice', 'count'),
    Average_USD=('usdprice', 'mean'),
    Median_USD=('usdprice', 'median'),
    Minimum_USD=('usdprice', 'min'),
    Maximum_USD=('usdprice', 'max')
).round(4)

**Note:** ZWL ki **1,267 rows** ka median USD-price sirf **0.15** hai jabki USD currency ki **56 rows** ka **1.265** (~8x fark). Fix: grouping mein `currency` bhi rakho (Part 2 mein yahi kiya) - is tarah ke fixes (currency-wise grouping + FX check) ke baad final cleaning mein sirf **632 rows (0.03%)** flag hui.

# Part 2 - Final Cleaning

Goal: KG food records rakhna, missing USD prices hatana, unusual values flag karna, aur SQL ke liye `retail` dataset banana.

### Cell 1 - Libraries aur output folder

Pandas data handle karta hai, NumPy numerical work ke liye hai, charts ke liye Seaborn/Matplotlib aur folders ke liye `os`.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

**Insight:** `outputs` folder baad mein Power BI CSV files save karega. Folder pehle se ho toh `exist_ok=True` error nahi aane deta.

### Cell 2 - Country names

Dataset ke country codes ko readable names mein convert karne ke liye dictionary.

In [ ]:
NAMES = {"AFG": "Afghanistan", "AGO": "Angola", "ARG": "Argentina", "ARM": "Armenia", "BDI": "Burundi", "BEN": "Benin", "BFA": "Burkina Faso", "BGD": "Bangladesh", "BLR": "Belarus", "BOL": "Bolivia", "BTN": "Bhutan", "CAF": "Central African Rep.", "CHN": "China", "CIV": "Cote d'Ivoire", "CMR": "Cameroon", "COD": "DR Congo", "COG": "Congo", "COL": "Colombia", "CPV": "Cabo Verde", "DJI": "Djibouti", "DOM": "Dominican Rep.", "DZA": "Algeria", "ECU": "Ecuador", "EGY": "Egypt", "ETH": "Ethiopia", "FJI": "Fiji", "GHA": "Ghana", "GIN": "Guinea", "GMB": "Gambia", "GNB": "Guinea-Bissau", "GTM": "Guatemala", "HND": "Honduras", "HTI": "Haiti", "IDN": "Indonesia", "IND": "India", "IRN": "Iran", "IRQ": "Iraq", "JOR": "Jordan", "JPN": "Japan", "KAZ": "Kazakhstan", "KEN": "Kenya", "KGZ": "Kyrgyzstan", "KHM": "Cambodia", "LAO": "Laos", "LBN": "Lebanon", "LBR": "Liberia", "LBY": "Libya", "LKA": "Sri Lanka", "LSO": "Lesotho", "MDA": "Moldova", "MDG": "Madagascar", "MEX": "Mexico", "MLI": "Mali", "MMR": "Myanmar", "MNG": "Mongolia", "MOZ": "Mozambique", "MRT": "Mauritania", "MWI": "Malawi", "NAM": "Namibia", "NER": "Niger", "NGA": "Nigeria", "NIC": "Nicaragua", "NPL": "Nepal", "PAK": "Pakistan", "PAN": "Panama", "PER": "Peru", "PHL": "Philippines", "PRY": "Paraguay", "PSE": "Palestine", "RUS": "Russia", "RWA": "Rwanda", "SDN": "Sudan", "SEN": "Senegal", "SLE": "Sierra Leone", "SLV": "El Salvador", "SOM": "Somalia", "SSD": "South Sudan", "SWZ": "Eswatini", "SYR": "Syria", "TCD": "Chad", "TGO": "Togo", "THA": "Thailand", "TJK": "Tajikistan", "TLS": "Timor-Leste", "TUR": "Turkey", "TZA": "Tanzania", "UGA": "Uganda", "UKR": "Ukraine", "VEN": "Venezuela", "VNM": "Vietnam", "YEM": "Yemen", "ZAF": "South Africa", "ZMB": "Zambia", "ZWE": "Zimbabwe"}

**Insight:** Analysis mein country ISO code use hoga. Dictionary sirf output/report mein country ka full name dikhane ke kaam aayegi.

### Cell 3 - Basic filtering

Sirf KG unit, food category aur available USD price wale records rakhenge.

In [ ]:
clean = combined_df[
    (combined_df["unit"] == "KG") &
    (combined_df["category"] != "non-food")
].dropna(subset=["usdprice"]).copy()

start_rows = len(clean)
print("Rows after basic filtering:", f"{start_rows:,}")

**Insight:** KG, litre aur pieces ko ek saath compare nahi karna chahiye. Isliye abhi sirf KG records rakhe hain. Missing USD price wali rows is USD-based analysis mein use nahi hongi.

### Cell 4 - Month aur year columns

Monthly aur yearly analysis ke liye date se do columns banayenge.

In [ ]:
clean["ym"] = clean["date"].dt.to_period("M").astype(str)
clean["year"] = clean["date"].dt.year

clean[["date", "ym", "year"]].head()

**Insight:** `ym` jaise `2024-05` monthly grouping ke liye hai. `year` yearly inflation nikalne mein use hoga.

### Cell 5 - FX consistency check

Local price ko USD price se divide karke FX proxy banayenge, phir same country/currency/month ke median se compare karenge.

In [ ]:
clean["fx"] = clean["price"] / clean["usdprice"]

fx_median = clean.groupby(
    ["countryiso3", "currency", "ym"]
)["fx"].transform("median")

clean["fx_ratio"] = clean["fx"] / fx_median

clean[["countryiso3", "currency", "ym", "fx", "fx_ratio"]].head()

**Insight:** `fx_ratio` 1 ke kareeb ho toh record apne group median ke kareeb hai. Bahut high/low value possible currency/data issue ho sakti hai, lekin automatically galat prove nahi hoti.

### Cell 6 - Price consistency check

Same country, commodity, currency aur year ke median price se har row compare karenge.

In [ ]:
price_median = clean.groupby(
    ["countryiso3", "commodity", "currency", "year"]
)["price"].transform("median")

clean["price_ratio"] = clean["price"] / price_median

clean[["countryiso3", "commodity", "year", "price", "price_ratio"]].head()

**Insight:** `price_ratio` 1 ka matlab price group median ke barabar hai. 20 se zyada ya 0.05 se kam value ko review ke liye flag karenge.

### Cell 7 - Unusual records flag karna

Thresholds ko alag variables mein rakha hai, taaki logic readable rahe.

In [ ]:
bad = (
    (clean["fx_ratio"] > 3) |
    (clean["fx_ratio"] < 1 / 3) |
    (clean["price_ratio"] > 20) |
    (clean["price_ratio"] < 0.05)
)

print("Rows checked:", f"{len(clean):,}")
print("Rows flagged:", f"{bad.sum():,}")
print("Flagged percentage:", f"{bad.mean() * 100:.3f}%")

**Insight:** Flagged records unusual hain, zaroori nahi sab incorrect hon. Genuine price shocks bhi extreme values bana sakte hain. Thresholds project ki methodology/limitation mein explain karna.

### Cell 8 - Final clean dataset aur Retail subset

Temporary calculation columns hata kar clean dataset banayenge. SQL ke liye Retail records alag rakhenge.

In [ ]:
clean = clean.loc[~bad].drop(
    columns=["fx", "fx_ratio", "price_ratio"]
).copy()

retail = clean[clean["pricetype"] == "Retail"].copy()

print("Final clean rows:", f"{len(clean):,}")
print("Retail rows:", f"{len(retail):,}")
print("Countries:", clean["countryiso3"].nunique())
print("Commodities:", clean["commodity"].nunique())

**Insight:** `clean` filtered food records hain. `retail` unhi records ka Retail-only subset hai. Neeche ke SQL questions `retail` par chalenge.

# Part 3 - SQL Business Analysis

Har SQL question alag cells mein hai. Pehle SQL se summary table banegi, phir alag query result nikalegi. Query ka output Pandas table mein dikhega.

### SQL setup - SQLite connection

In [ ]:
import sqlite3

con = sqlite3.connect(":memory:")

prices = retail[
    ["countryiso3", "market_id", "admin1", "commodity",
     "category", "currency", "year", "price", "usdprice",
     "ym", "date"]
].copy()

prices["month_no"] = (
    prices["year"] * 12 +
    pd.to_datetime(prices["date"]).dt.month
)

prices.to_sql("prices", con, index=False, if_exists="replace")

pd.read_sql(
    "SELECT COUNT(*) AS total_rows, "
    "COUNT(DISTINCT countryiso3) AS countries, "
    "COUNT(DISTINCT commodity) AS commodities FROM prices",
    con
)

**Insight:** `prices` SQL table ready hai. `month_no` consecutive months ko identify karne mein help karega.

## SQL Question 1 - Year-over-year food inflation

**Business question:** Ek commodity ka average price pichle saal ke comparison mein kitna badla?

### Q1 - Step 1: Yearly average price table

In [ ]:
sql = """
CREATE TABLE yearly AS
SELECT countryiso3 AS country, commodity, currency, year,
       AVG(price) AS avg_price,
       COUNT(DISTINCT ym) AS months
FROM prices
GROUP BY countryiso3, commodity, currency, year
HAVING COUNT(DISTINCT ym) >= 6
"""
con.execute("DROP TABLE IF EXISTS yearly")
con.execute(sql)
pd.read_sql("SELECT * FROM yearly LIMIT 5", con)

**Insight:** `yearly` table mein har country + commodity + currency + year ka average price hai. Kam se kam 6 months ka data rakha hai, taaki bahut kam observations par yearly average na bane.

### Q1 - Step 2: LAG se previous year price

In [ ]:
sql = """
CREATE TABLE yoy AS
SELECT country, commodity, currency, year,
       (avg_price - prev_price) * 100.0 / prev_price AS inflation_pct
FROM (
    SELECT *,
           LAG(avg_price) OVER (
               PARTITION BY country, commodity, currency ORDER BY year
           ) AS prev_price,
           LAG(year) OVER (
               PARTITION BY country, commodity, currency ORDER BY year
           ) AS prev_year
    FROM yearly
)
WHERE prev_year = year - 1
"""
con.execute("DROP TABLE IF EXISTS yoy")
con.execute(sql)
pd.read_sql("SELECT * FROM yoy LIMIT 5", con)

**Insight:** `LAG()` previous available year ka average price deta hai. `prev_year = year - 1` se sirf consecutive years compare hote hain.

### Q1 - Step 3: Year-wise average inflation

In [ ]:
q1_year = """
SELECT year, ROUND(AVG(inflation_pct), 1) AS avg_inflation_pct
FROM yoy
WHERE inflation_pct < 1000
GROUP BY year
ORDER BY year
"""
display(pd.read_sql(q1_year, con))

**Insight:** Is output se pata chalega kis year mein average YoY food inflation sabse zyada thi. 1000% se upar readings ko is summary se exclude kiya gaya hai, kyunki kuch cases currency rescaling se affected ho sakte hain.

### Q1 - Step 4: 2025 ke top inflation countries

In [ ]:
q1_country = """
SELECT country,
       ROUND(AVG(inflation_pct), 1) AS avg_inflation_pct,
       COUNT(*) AS commodities
FROM yoy
WHERE year = 2025 AND inflation_pct < 1000
GROUP BY country
HAVING COUNT(*) >= 5
ORDER BY avg_inflation_pct DESC
LIMIT 10
"""
r1 = pd.read_sql(q1_country, con)
r1["country_name"] = r1["country"].map(NAMES)
display(r1)

**Insight:** Ye countries dikhate hain jahan 2025 mein multiple commodities par inflation high rahi. `HAVING COUNT(*) >= 5` minimum 5 commodity observations require karta hai.

### Q1 - Step 5: 2020 se 2025 commodity price rise

In [ ]:
q1_commodity = """
SELECT a.commodity,
       ROUND(AVG((b.avg_price - a.avg_price) * 100.0 / a.avg_price), 1)
           AS price_rise_pct,
       COUNT(*) AS series
FROM yearly a
JOIN yearly b
  ON a.country = b.country
 AND a.commodity = b.commodity
 AND a.currency = b.currency
WHERE a.year = 2020
  AND b.year = 2025
  AND b.avg_price < a.avg_price * 10
GROUP BY a.commodity
HAVING COUNT(*) >= 8
ORDER BY price_rise_pct DESC
"""
r2 = pd.read_sql(q1_commodity, con)
display(r2.head(10))

**Insight:** Query same country, commodity aur currency ke 2020 aur 2025 averages compare karti hai. Result mein top commodities ka long-term price rise dikhega. `series >= 8` ka matlab kam se kam 8 comparable country-series hon.

## SQL Question 2 - Monthly price volatility

**Business question:** Kis category, commodity aur country mein prices month-to-month zyada change hote hain?

### Q2 - Step 1: Market-wise monthly average

In [ ]:
sql = """
CREATE TABLE monthly_market AS
SELECT countryiso3 AS country, market_id, commodity, category,
       currency, month_no, AVG(price) AS price
FROM prices
GROUP BY countryiso3, market_id, commodity, category, currency, month_no
"""
con.execute("DROP TABLE IF EXISTS monthly_market")
con.execute(sql)
pd.read_sql("SELECT * FROM monthly_market LIMIT 5", con)

**Insight:** Ek market mein same commodity ke same month ke multiple records ka average liya. Isse month-wise comparison simple hota hai.

### Q2 - Step 2: LAG se month-to-month swing

In [ ]:
sql = """
CREATE TABLE swings AS
SELECT country, commodity, category,
       ABS(price - prev_price) * 100.0 / prev_price AS swing_pct
FROM (
    SELECT *,
           LAG(price) OVER (
               PARTITION BY country, market_id, commodity, currency
               ORDER BY month_no
           ) AS prev_price,
           LAG(month_no) OVER (
               PARTITION BY country, market_id, commodity, currency
               ORDER BY month_no
           ) AS prev_month
    FROM monthly_market
)
WHERE prev_month = month_no - 1
"""
con.execute("DROP TABLE IF EXISTS swings")
con.execute(sql)
pd.read_sql("SELECT * FROM swings LIMIT 5", con)

**Insight:** `ABS()` price increase aur decrease dono ko positive change ke roop mein measure karta hai. `prev_month = month_no - 1` ensure karta hai ki gap wale months compare na hon.

### Q2 - Step 3: Category-wise volatility

In [ ]:
q2_category = """
SELECT category,
       ROUND(AVG(swing_pct), 1) AS avg_monthly_swing_pct
FROM swings
WHERE swing_pct < 300
GROUP BY category
ORDER BY avg_monthly_swing_pct DESC
"""
display(pd.read_sql(q2_category, con))

**Insight:** Higher percentage ka matlab category ke prices mein average month-to-month movement zyada hai. 300% ya usse zyada swings ko summary se filter kiya hai.

### Q2 - Step 4: Commodity-wise volatility

In [ ]:
q2_commodity = """
SELECT commodity,
       ROUND(AVG(swing_pct), 1) AS avg_monthly_swing_pct,
       COUNT(*) AS observations
FROM swings
WHERE swing_pct < 300
GROUP BY commodity
HAVING COUNT(*) >= 2000
ORDER BY avg_monthly_swing_pct DESC
"""
r3 = pd.read_sql(q2_commodity, con)
display(r3.head(10))

**Insight:** Is ranking se high-volatility commodities identify hoti hain. Minimum observations condition low-sample commodities ko ranking se door rakhti hai.

### Q2 - Step 5: Country-wise volatility

In [ ]:
q2_country = """
SELECT country,
       ROUND(AVG(swing_pct), 1) AS avg_monthly_swing_pct,
       COUNT(*) AS observations
FROM swings
WHERE swing_pct < 300
GROUP BY country
HAVING COUNT(*) >= 2000
ORDER BY avg_monthly_swing_pct DESC
"""
r4 = pd.read_sql(q2_country, con)
r4["country_name"] = r4["country"].map(NAMES)
display(r4.head(10))

**Insight:** Ye batata hai kis country mein sampled market prices zyada fluctuate karte hain. Ye market coverage aur available observations se bhi affect ho sakta hai.

## SQL Question 3 - Regional price gap

**Business question:** Ek hi country mein same commodity ka price regions ke beech kitna differ karta hai?

### Q3 - Step 1: Region-wise monthly average price

In [ ]:
sql = """
CREATE TABLE region_prices AS
SELECT countryiso3 AS country, commodity, currency, ym,
       admin1 AS region, AVG(price) AS price
FROM prices
WHERE admin1 <> 'Unknown'
GROUP BY countryiso3, commodity, currency, ym, admin1
"""
con.execute("DROP TABLE IF EXISTS region_prices")
con.execute(sql)
pd.read_sql("SELECT * FROM region_prices LIMIT 5", con)

**Insight:** Unknown region exclude kiya hai, kyunki usse actual regional comparison nahi ho sakta.

### Q3 - Step 2: Country-commodity-month price gap

In [ ]:
sql = """
CREATE TABLE gap AS
SELECT country, commodity, currency, ym,
       AVG(price) AS avg_price,
       (MAX(price) - MIN(price)) * 100.0 / AVG(price) AS gap_pct
FROM region_prices
GROUP BY country, commodity, currency, ym
HAVING COUNT(*) >= 3
"""
con.execute("DROP TABLE IF EXISTS gap")
con.execute(sql)
pd.read_sql("SELECT * FROM gap LIMIT 5", con)

**Insight:** Gap percentage max aur min regional price ke difference ko average regional price ke percentage mein dikhata hai. Minimum 3 regions required hain.

### Q3 - Step 3: Country-wise average regional gap

In [ ]:
q3_country = """
SELECT country, ROUND(AVG(gap_pct), 1) AS avg_gap_pct,
       COUNT(*) AS observations
FROM gap
GROUP BY country
HAVING COUNT(*) >= 100
ORDER BY avg_gap_pct DESC
"""
r5 = pd.read_sql(q3_country, con)
r5["country_name"] = r5["country"].map(NAMES)
display(r5.head(10))

**Insight:** High gap ka matlab same food item ke prices regions ke beech zyada differ karte hain. Ye transport cost, access aur local supply differences se related ho sakta hai.

### Q3 - Step 4: Commodity-wise regional gap

In [ ]:
q3_commodity = """
SELECT commodity, ROUND(AVG(gap_pct), 1) AS avg_gap_pct,
       COUNT(DISTINCT country) AS countries
FROM gap
GROUP BY commodity
HAVING COUNT(DISTINCT country) >= 8
ORDER BY avg_gap_pct DESC
LIMIT 10
"""
display(pd.read_sql(q3_commodity, con))

**Insight:** Isse pata chalta hai kaunse commodities ke prices regions ke beech consistently zyada vary karte hain.

### Q3 - Step 5: Regions with highest premium

In [ ]:
q3_region = """
SELECT r.country, r.region,
       ROUND(AVG(r.price * 100.0 / g.avg_price) - 100, 1) AS premium_pct,
       COUNT(*) AS observations
FROM region_prices r
JOIN gap g
  ON r.country = g.country
 AND r.commodity = g.commodity
 AND r.currency = g.currency
 AND r.ym = g.ym
GROUP BY r.country, r.region
HAVING COUNT(*) >= 200
ORDER BY premium_pct DESC
LIMIT 10
"""
display(pd.read_sql(q3_region, con))

**Insight:** Positive premium batata hai region ka average price overall country-commodity-month average se kitna upar hai.

## SQL Question 4 - USD price index (2020 = 100)

**Business question:** 2020 ke comparison mein food prices ka USD level baad ke months mein kaise badla?

### Q4 - Step 1: 2020 base price

In [ ]:
sql = """
CREATE TABLE base_2020 AS
SELECT countryiso3 AS country, commodity, currency,
       AVG(usdprice) AS base_price
FROM prices
WHERE year = 2020
GROUP BY countryiso3, commodity, currency
"""
con.execute("DROP TABLE IF EXISTS base_2020")
con.execute(sql)
pd.read_sql("SELECT * FROM base_2020 LIMIT 5", con)

**Insight:** Har country-commodity-currency series ka 2020 average USD price base banega. Sirf un series ka index calculate hoga jinka 2020 base available hai.

### Q4 - Step 2: Monthly USD price index

In [ ]:
sql = """
CREATE TABLE usd_index AS
SELECT p.countryiso3 AS country, p.commodity, p.category,
       p.year, p.ym,
       AVG(p.usdprice) * 100.0 / b.base_price AS idx
FROM prices p
JOIN base_2020 b
  ON p.countryiso3 = b.country
 AND p.commodity = b.commodity
 AND p.currency = b.currency
GROUP BY p.countryiso3, p.commodity, p.category,
         p.currency, p.year, p.ym
"""
con.execute("DROP TABLE IF EXISTS usd_index")
con.execute(sql)
pd.read_sql("SELECT * FROM usd_index LIMIT 5", con)

**Insight:** Index 100 ka matlab 2020 base ke barabar. 120 ka matlab base se 20% upar. 90 ka matlab base se 10% neeche.

### Q4 - Step 3: Category-wise index by month

In [ ]:
q4_category = """
SELECT category, ym, ROUND(AVG(idx), 1) AS avg_index
FROM usd_index
WHERE idx < 1000
GROUP BY category, ym
ORDER BY category, ym
"""
r6 = pd.read_sql(q4_category, con)
display(r6.head(15))

**Insight:** Monthly index trend se categories ke price levels compare kar sakte hain. Extreme index values ko summary se filter kiya gaya hai.

### Q4 - Step 4: Staples price jump from 2021 to 2022

In [ ]:
q4_staples = """
SELECT a.country,
       ROUND((b.v - a.v) * 100.0 / a.v, 1) AS jump_pct
FROM (
    SELECT country, AVG(idx) AS v
    FROM usd_index
    WHERE commodity IN ('Wheat flour', 'Rice', 'Maize', 'Sugar')
      AND year = 2021 AND idx < 1000
    GROUP BY country
) a
JOIN (
    SELECT country, AVG(idx) AS v
    FROM usd_index
    WHERE commodity IN ('Wheat flour', 'Rice', 'Maize', 'Sugar')
      AND year = 2022 AND idx < 1000
    GROUP BY country
) b
ON a.country = b.country
WHERE a.country NOT IN ('ZWE', 'SYR')
ORDER BY jump_pct DESC
"""
r7 = pd.read_sql(q4_staples, con)
r7["country_name"] = r7["country"].map(NAMES)
display(r7.head(10))
print("Countries compared:", len(r7))
print("Median jump:", round(r7["jump_pct"].median(), 1), "%")

**Insight:** Query 2021 aur 2022 ke staple-food index ka difference compare karti hai. Median jump ek typical country ka change samajhne mein average se zyada robust ho sakta hai.

## SQL Question 5 - Food Security Risk Index

**Business question:** Kaunse countries mein inflation, monthly volatility aur regional price gap ek saath high hain?

Score weights: inflation 40%, volatility 30%, regional gap 30%. Ye project-defined weights hain, official risk standard nahi.

### Q5 - Risk score query

In [ ]:
q5_risk = """
WITH infl AS (
    SELECT country, AVG(inflation_pct) AS inflation
    FROM yoy
    WHERE year = 2025 AND inflation_pct < 1000
    GROUP BY country
    HAVING COUNT(*) >= 5
),
vol AS (
    SELECT country, AVG(swing_pct) AS volatility
    FROM swings
    WHERE swing_pct < 300
    GROUP BY country
    HAVING COUNT(*) >= 2000
),
gp AS (
    SELECT country, AVG(gap_pct) AS regional_gap
    FROM gap
    WHERE ym >= '2024-01'
    GROUP BY country
    HAVING COUNT(*) >= 50
),
joined AS (
    SELECT infl.country, inflation, volatility, regional_gap
    FROM infl
    JOIN vol USING (country)
    JOIN gp USING (country)
),
ranked AS (
    SELECT *,
           RANK() OVER (ORDER BY inflation) * 1.0 / COUNT(*) OVER () AS r_infl,
           RANK() OVER (ORDER BY volatility) * 1.0 / COUNT(*) OVER () AS r_vol,
           RANK() OVER (ORDER BY regional_gap) * 1.0 / COUNT(*) OVER () AS r_gap
    FROM joined
)
SELECT country,
       ROUND(inflation, 1) AS inflation_2025,
       ROUND(volatility, 1) AS volatility,
       ROUND(regional_gap, 1) AS regional_gap,
       ROUND((0.4*r_infl + 0.3*r_vol + 0.3*r_gap) * 100, 1) AS risk_score,
       CASE
           WHEN (0.4*r_infl + 0.3*r_vol + 0.3*r_gap) * 100 >= 70 THEN 'High'
           WHEN (0.4*r_infl + 0.3*r_vol + 0.3*r_gap) * 100 >= 40 THEN 'Moderate'
           ELSE 'Low'
       END AS tier
FROM ranked
ORDER BY risk_score DESC
"""
sql_rk = pd.read_sql(q5_risk, con)
sql_rk["country_name"] = sql_rk["country"].map(NAMES)
display(sql_rk.head(10))
display(sql_rk.tail(5))
print("Countries scored:", len(sql_rk))
print("Risk tiers:", sql_rk["tier"].value_counts().to_dict())

**Insight:** Sirf wahi countries score hui hain jinke paas teeno measures available hain. High score ka matlab is project ke selected indicators aur weights ke hisaab se higher relative risk hai. Isse official food-security classification nahi samajhna chahiye.

## Final Summary

- Food inflation ko year-over-year compare kiya.
- Month-to-month price movement se volatility dekhi.
- Regions ke beech same commodity ka price gap calculate kiya.
- 2020 = 100 USD index banaya.
- Inflation, volatility aur regional gap ko combine karke project risk score banaya.

**Limitations:** Analysis sirf KG unit aur Retail price records par based hai. Country coverage har saal same nahi. Outlier thresholds aur risk weights project choices hain. Average values extreme records se affect ho sakti hain.

# Part 4 - Power BI Export

Neeche ka export cell SQL results ko CSV files mein save karta hai. Pehle Part 2 aur Part 3 ke saare cells run hone chahiye.

In [ ]:
PB = f"{OUT}/powerbi"; os.makedirs(PB, exist_ok=True)
def save(name, df):
    df.to_csv(f"{PB}/{name}.csv", index=False)
    print(f"{name:32s} rows: {len(df):,}")

def add_name(df, col="country"):
    df = df.copy(); df["country_name"] = df[col].map(NAMES); return df

# ---------- Facts (sab SQL se) ----------
save("fact_country_inflation", add_name(pd.read_sql("""
    SELECT country, year, ROUND(AVG(inflation_pct),1) AS avg_inflation_pct, COUNT(*) AS commodities
    FROM yoy WHERE inflation_pct < 1000 GROUP BY country, year""", con)))

save("fact_commodity_inflation", pd.read_sql("""
    SELECT a.commodity, ROUND(AVG((b.avg_price - a.avg_price)*100.0/a.avg_price),1) AS price_rise_pct, COUNT(*) AS series
    FROM yearly a JOIN yearly b ON a.country=b.country AND a.commodity=b.commodity AND a.currency=b.currency
    WHERE a.year=2020 AND b.year=2025 AND b.avg_price < a.avg_price*10
    GROUP BY a.commodity HAVING COUNT(*) >= 8""", con))

save("fact_volatility_category", pd.read_sql("""
    SELECT category, ROUND(AVG(swing_pct),1) AS avg_monthly_swing_pct, COUNT(*) AS observations
    FROM swings WHERE swing_pct < 300 GROUP BY category""", con))

save("fact_volatility_commodity", pd.read_sql("""
    SELECT commodity, category, ROUND(AVG(swing_pct),1) AS avg_monthly_swing_pct, COUNT(*) AS observations
    FROM swings WHERE swing_pct < 300 GROUP BY commodity, category HAVING COUNT(*) >= 2000""", con))

save("fact_volatility_country", add_name(pd.read_sql("""
    SELECT country, ROUND(AVG(swing_pct),1) AS avg_monthly_swing_pct, COUNT(*) AS observations
    FROM swings WHERE swing_pct < 300 GROUP BY country HAVING COUNT(*) >= 2000""", con)))

save("fact_regional_gap_country", add_name(pd.read_sql("""
    SELECT country, ROUND(AVG(gap_pct),1) AS avg_gap_pct, COUNT(*) AS observations
    FROM gap GROUP BY country HAVING COUNT(*) >= 100""", con)))

save("fact_regional_gap_commodity", pd.read_sql("""
    SELECT commodity, ROUND(AVG(gap_pct),1) AS avg_gap_pct, COUNT(DISTINCT country) AS countries
    FROM gap GROUP BY commodity HAVING COUNT(DISTINCT country) >= 8""", con))

save("fact_region_premium", add_name(pd.read_sql("""
    SELECT r.country, r.region, ROUND(AVG(r.price*100.0/g.avg_price)-100,1) AS premium_pct, COUNT(*) AS observations
    FROM region_prices r JOIN gap g
      ON r.country=g.country AND r.commodity=g.commodity AND r.currency=g.currency AND r.ym=g.ym
    GROUP BY r.country, r.region HAVING COUNT(*) >= 200""", con)))

save("fact_usd_index", pd.read_sql("""
    SELECT category, ym, ROUND(AVG(idx),1) AS avg_index
    FROM usd_index WHERE idx < 1000 GROUP BY category, ym""", con))

save("fact_shock_2022", add_name(r7[["country","jump_pct"]]))
save("fact_country_risk", add_name(sql_rk.drop(columns=["name"], errors="ignore")))

# ---------- Dimensions ----------
dim_country = pd.read_sql("SELECT DISTINCT countryiso3 AS country FROM prices", con)
dim_country["country_name"] = dim_country.country.map(NAMES)
dim_country = dim_country.merge(sql_rk[["country","tier"]], on="country", how="left")
dim_country["tier"] = dim_country.tier.fillna("Not scored")
save("dim_country", dim_country)

save("dim_commodity", pd.read_sql("""
    SELECT commodity, category FROM prices GROUP BY commodity, category""", con).drop_duplicates("commodity"))

dim_date = pd.read_sql("SELECT DISTINCT ym, year FROM prices ORDER BY ym", con)
dim_date["month_no"] = dim_date.ym.str[5:7].astype(int)
dim_date["month"] = pd.to_datetime(dim_date.ym + "-01").dt.strftime("%b")
save("dim_date", dim_date)

print("\nFiles:", sorted(os.listdir(PB)))